In [1]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torchvision.models import VGG16_Weights
from torchvision import datasets

import os
import time
import torchvision.transforms.functional as TF

from torch.utils.data import random_split, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

Using: cuda


In [2]:
#pull the pretrained vgg model and change the last classifier to handle 4 classes 
#(snow, mixed, ice, dry)
model = models.vgg16(weights=VGG16_Weights.DEFAULT)
model.classifier[6] = nn.Linear(4096, 4)

#the vgg model contains millions of params, turn all fo them off :)
for param in model.parameters():
    param.requires_grad = False

#only turn on the params for our classifier, its better for image classification
for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(device)
print("Model Defined")

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 214MB/s]


Model Defined


In [3]:
class RoadDataset(datasets.ImageFolder):
    def __getitem__(self, index):
        # 1. Get the path, image, and label
        path, label_idx = self.samples[index]
        image = self.loader(path) # Loads as PIL Image
        image = image.convert('RGB')

        filename = os.path.basename(path)
        condition = self.classes[label_idx]

        width, height = image.size

        if "arenacam" in filename:
            # Car hood in image
            image = TF.crop(image, top=700, left=1200, height=400, width=400)
        elif "pylon_camera" in filename:
            # No car hood in view
            image = TF.crop(image, top=400, left=800, height=400, width=400)
        elif width < 400:
            pass #dont crop images that are too small
        else:
            #base case just in case
            image = TF.center_crop(image, (400, 400))

        # apply transforms (resize, tensor, normalize)
        if self.transform is not None:
            image = self.transform(image)

        # Returns: (image, index of label)
        return image, label_idx

    def get_classes(self):
        return self.classes
        
print("Data loader Defined")

Data loader Defined


In [4]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    base_path = '/content/drive/Shareddrives/WSPI/catergorized2'
except Exception:
    if os.path.exists('/kaggle'):
        base_path = '/kaggle/input/datasets/johnson183/catergorized2/catergorized2'
    else:
        base_path = './data/images'

print("Input Defined")

Input Defined


In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = RoadDataset(root=base_path, transform=transform)
class_names = full_dataset.get_classes()
print(class_names)

train_size = int(0.9 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_subset, test_subset = random_split(full_dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False, num_workers=2)

print("Dataset Defined")

['dry', 'ice', 'mixed', 'snow']
Dataset Defined


In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr = 0.0001) # Reduced to 0.0001 to stabilize training

model.train()

#num of times we train the model
num_epochs = 10
#train!
for epoch in range(num_epochs):
    print(f'Training epoch {epoch+1}...')
    start = time.time() # Time each epoch
    
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    elapsed = time.time() - start
    print(f'Epoch {epoch+1} | Loss: {running_loss / len(train_loader):.4f} | Time: {elapsed:.1f}s\n')

Training epoch 1...
Epoch 1 | Loss: 0.1037 | Time: 268.9s

Training epoch 2...
Epoch 2 | Loss: 0.0377 | Time: 213.5s

Training epoch 3...
Epoch 3 | Loss: 0.0308 | Time: 210.5s

Training epoch 4...
Epoch 4 | Loss: 0.0269 | Time: 223.3s

Training epoch 5...
Epoch 5 | Loss: 0.0224 | Time: 216.0s

Training epoch 6...
Epoch 6 | Loss: 0.0211 | Time: 219.1s

Training epoch 7...
Epoch 7 | Loss: 0.0204 | Time: 218.2s

Training epoch 8...
Epoch 8 | Loss: 0.0272 | Time: 216.7s

Training epoch 9...
Epoch 9 | Loss: 0.0377 | Time: 217.6s

Training epoch 10...
Epoch 10 | Loss: 0.0300 | Time: 218.0s



In [7]:
def evaluate(loader, dataName=""):
    correct = 0
    total = 0
    class_correct = [0] * len(class_names)
    class_total = [0] * len(class_names)

    model.eval()
    with torch.no_grad():
        for data in loader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            for i in range(len(labels)):
                label_idx = labels[i].item()
                class_total[label_idx] += 1
                
                if predicted[i] == labels[i]:
                    class_correct[label_idx] += 1

    print(f"\n{dataName} Results:")
    print(f"Overall: {correct}/{total}  Accuracy: {100 * correct / total:.1f}%")
    for i, name in enumerate(class_names):
        acc = 100 * class_correct[i] / class_total[i] if class_total[i] > 0 else 0
        print(f"  {name}: {class_correct[i]}/{class_total[i]}  Accuracy: {acc:.1f}%")

evaluate(test_loader, "Test Set")

cv_path = '/kaggle/input/datasets/johnson183/cv-images/cv_images'
cv_dataset = RoadDataset(root=cv_path, transform=transform)
cv_loader = DataLoader(cv_dataset, batch_size=32, shuffle=False, num_workers=2)
evaluate(cv_loader, "External CV Images")


Test Set Results:
Overall: 791/804  Accuracy: 98.4%
  dry: 76/78  Accuracy: 97.4%
  ice: 75/75  Accuracy: 100.0%
  mixed: 54/61  Accuracy: 88.5%
  snow: 586/590  Accuracy: 99.3%

External CV Images Results:
Overall: 22/60  Accuracy: 36.7%
  dry: 1/15  Accuracy: 6.7%
  ice: 7/15  Accuracy: 46.7%
  mixed: 1/15  Accuracy: 6.7%
  snow: 13/15  Accuracy: 86.7%
